In [1]:
import polars as pl
import datetime
from polars import DataFrame

from etl.aml.utils import *

In [4]:
dataset = '../../data/raw/HI-Medium_Trans.csv'

trans = (
    pl.read_csv(dataset)
    .with_columns(
        pl.col("Timestamp").str.strptime(pl.Datetime, '%Y/%m/%d %H:%M', strict=True)
    )
    .sort('Timestamp')
)

In [5]:
trans.head()

Timestamp,From Bank,From,To Bank,To,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
datetime[μs],i64,str,i64,str,f64,str,f64,str,str,i64
2022-09-01 00:00:00,1046,"""800A37D90""",274159,"""820C04F20""",26.42,"""US Dollar""",26.42,"""US Dollar""","""Credit Card""",0
2022-09-01 00:00:00,21418,"""800AB4DE0""",21418,"""800AB4DE0""",23.61,"""US Dollar""",23.61,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,32248,"""80056BBD0""",11,"""800BAFE20""",12373.74,"""US Dollar""",12373.74,"""US Dollar""","""ACH""",0
2022-09-01 00:00:00,11798,"""80145BAF0""",11798,"""80145BAF0""",4981.27,"""US Dollar""",4981.27,"""US Dollar""","""Reinvestment""",0
2022-09-01 00:00:00,1924,"""800CCE420""",1924,"""800CCE420""",22.94,"""US Dollar""",22.94,"""US Dollar""","""Reinvestment""",0


In [6]:
zero = trans['Timestamp'].min()
hundred = trans['Timestamp'].max()

In [7]:
diff = hundred - zero
days = diff.days
sixty = zero + datetime.timedelta(days=days * 0.6)
eighty = zero + datetime.timedelta(days=days * 0.8)
hundred = zero + datetime.timedelta(days=days)

print(zero, sixty, eighty, hundred)

2022-09-01 00:00:00 2022-09-17 04:48:00 2022-09-22 14:24:00 2022-09-28 00:00:00


# Convert currencies

In [8]:
trans = convert_currencies_to_usd(trans)

In [9]:
trans.head()

Timestamp,From Bank,From,To Bank,To,Payment Format,Is Laundering,Amount
datetime[μs],i64,str,i64,str,str,i64,f64
2022-09-01 00:00:00,1046,"""800A37D90""",274159,"""820C04F20""","""Credit Card""",0,26.42
2022-09-01 00:00:00,21418,"""800AB4DE0""",21418,"""800AB4DE0""","""Reinvestment""",0,23.61
2022-09-01 00:00:00,32248,"""80056BBD0""",11,"""800BAFE20""","""ACH""",0,12373.74
2022-09-01 00:00:00,11798,"""80145BAF0""",11798,"""80145BAF0""","""Reinvestment""",0,4981.27
2022-09-01 00:00:00,1924,"""800CCE420""",1924,"""800CCE420""","""Reinvestment""",0,22.94


# Format data to remove strings

In [10]:
ssl = remove_strings(trans)

In [11]:
ssl.head()

Timestamp,From Bank,To Bank,Is Laundering,Amount,From,To,Payment Format
datetime[μs],i64,i64,i64,f64,u32,u32,u32
2022-09-01 00:00:00,1046,274159,0,26.42,1997667,581807,0
2022-09-01 00:00:00,21418,21418,0,23.61,1755234,1755234,6
2022-09-01 00:00:00,32248,11,0,12373.74,1077771,2067545,5
2022-09-01 00:00:00,11798,11798,0,4981.27,552140,552140,6
2022-09-01 00:00:00,1924,1924,0,22.94,519381,519381,6


# Global vars

In [12]:
every = '2d'

In [13]:
def _prep(df: pl.DataFrame) -> pl.LazyFrame:
    return (
        df.lazy()
        .with_columns(
            pl.col('Timestamp').dt.truncate(every).alias("window_start")
        )
        .filter(
             pl.col('Timestamp').is_not_null()
             & pl.col('From').is_not_null()
             & pl.col('To').is_not_null()
        )
    )

In [14]:
_prep(ssl).sort(pl.col('window_start')).collect()

Timestamp,From Bank,To Bank,Is Laundering,Amount,From,To,Payment Format,window_start
datetime[μs],i64,i64,i64,f64,u32,u32,u32,datetime[μs]
2022-09-01 00:00:00,1046,274159,0,26.42,1997667,581807,0,2022-09-01 00:00:00
2022-09-01 00:00:00,21418,21418,0,23.61,1755234,1755234,6,2022-09-01 00:00:00
2022-09-01 00:00:00,32248,11,0,12373.74,1077771,2067545,5,2022-09-01 00:00:00
2022-09-01 00:00:00,11798,11798,0,4981.27,552140,552140,6,2022-09-01 00:00:00
2022-09-01 00:00:00,1924,1924,0,22.94,519381,519381,6,2022-09-01 00:00:00
…,…,…,…,…,…,…,…,…
2022-09-28 11:59:00,211767,219853,1,1050.049527,1529885,699788,5,2022-09-27 00:00:00
2022-09-28 12:14:00,211767,211767,0,5825.694684,1529885,1529885,5,2022-09-27 00:00:00
2022-09-28 12:14:00,211767,130596,1,5825.694684,1529885,1182383,5,2022-09-27 00:00:00


# Temporal stats

In [15]:
def add_temporal_stats(df: pl.DataFrame) -> pl.DataFrame:
    df = _prep(df).select(
        pl.col('window_start'),
        pl.col('Timestamp'),
        pl.col('From'),
        pl.col('To'),
    )

    in_gaps = (
        df.sort(["window_start", "To", "Timestamp"])
        .group_by(["window_start", "To"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_in_sec"),
            pl.col("gap").list.var().alias("var_gap_in_sec"),
        ])
        .select(["window_start", 'To', "mu_gap_in_sec", "var_gap_in_sec"])
        .rename({'To': 'Node'})
    )

    out_gaps = (
        df.sort(["window_start", "From", "Timestamp"])
        .group_by(["window_start", "From"])
        .agg(
            pl.col('Timestamp').diff().dt.total_seconds().alias('gap')
        )
        .with_columns(
            pl.col("gap").list.drop_nulls().alias("gap")
        )
        .with_columns([
            pl.col("gap").list.mean().alias("mu_gap_out_sec"),
            pl.col("gap").list.var().alias("var_gap_out_sec"),
        ])
        .select(["window_start", 'From', "mu_gap_out_sec", "var_gap_out_sec"])
        .rename({'From': 'Node'})
    )

    return (
        in_gaps.join(out_gaps, on=["Node", 'window_start'], how="full", coalesce=True)
        .fill_null(-1)
        .collect()
    )

In [16]:
temporal_features = add_temporal_stats(ssl)

In [17]:
temporal_features

window_start,Node,mu_gap_in_sec,var_gap_in_sec,mu_gap_out_sec,var_gap_out_sec
datetime[μs],i64,f64,f64,f64,f64
2022-09-13 00:00:00,742884,15720.0,6.8859e8,-1.0,-1.0
2022-09-01 00:00:00,197269,16894.285714,3.1682e8,-1.0,-1.0
2022-09-01 00:00:00,48866,36540.0,2.0499e9,9409.411765,2.7515e8
2022-09-01 00:00:00,285421,18408.0,6.2081532e8,360.0,-1.0
2022-09-07 00:00:00,748479,16971.428571,4.7460e8,-1.0,-1.0
…,…,…,…,…,…
2022-09-11 00:00:00,1358908,-1.0,-1.0,-1.0,-1.0
2022-09-11 00:00:00,1062340,-1.0,-1.0,90.0,16200.0
2022-09-01 00:00:00,2390,-1.0,-1.0,-1.0,-1.0


# Fraudulent patterns

## 2-cycles

In [18]:
def count_reciprocal_neighbours(df: DataFrame) -> DataFrame:
    lf = _prep(df)

    edges = (
        lf.select(["window_start", "From", "To"])
        .filter(pl.col("From") != pl.col("To"))
        .group_by(["window_start", "From", "To"])
        .agg(pl.len().alias("w"))
    )

    recip = (
        edges.join(
            edges,
            left_on=["window_start", "From", "To"],
            right_on=["window_start", "To", "From"],
            how="inner",
            suffix='_rev',
        )
        .with_columns(
            pl.min_horizontal("w", "w_rev").alias("cycle_count")
        )
        .group_by(["window_start", "From"])
        .agg(
            pl.col("cycle_count").sum().alias("r_2cycle")
        )
        .rename({"From": "Node"})
    )

    nodes = (
        pl.concat([
            lf.select(["window_start", pl.col("From").alias("Node")]),
            lf.select(["window_start", pl.col("To").alias("Node")]),
        ])
        .unique()
    )

    return (
        nodes.join(recip, on=["window_start", "Node"], how="left")
        .with_columns(
            pl.col("r_2cycle").fill_null(0).cast(pl.Int64)
        )
        .collect()
    )

In [19]:
two_cycles = count_reciprocal_neighbours(ssl)

In [20]:
two_cycles

window_start,Node,r_2cycle
datetime[μs],u32,i64
2022-09-09 00:00:00,273553,0
2022-09-01 00:00:00,1887036,0
2022-09-03 00:00:00,1786819,0
2022-09-05 00:00:00,77713,0
2022-09-15 00:00:00,1203669,0
…,…,…
2022-09-15 00:00:00,632559,0
2022-09-15 00:00:00,1133837,0
2022-09-01 00:00:00,1028539,0


In [21]:
two_cycles.describe()

statistic,window_start,Node,r_2cycle
str,str,f64,f64
"""count""","""9337964""",9.337964e6,9.337964e6
"""null_count""","""0""",0.0,0.0
"""mean""","""2022-09-07 18:20:54.673845""",1.0386e6,0.001269
"""std""",null,599475.946572,0.038422
"""min""","""2022-09-01 00:00:00""",0.0,0.0
"""25%""","""2022-09-03 00:00:00""",519274.0,0.0
"""50%""","""2022-09-09 00:00:00""",1.038702e6,0.0
"""75%""","""2022-09-13 00:00:00""",1.557395e6,0.0
"""max""","""2022-09-27 00:00:00""",2.076998e6,7.0


## Ego profile

In [22]:
EPS = 1e-12

def compute_ego_profiles(trans: DataFrame) -> DataFrame:
    lf = _prep(trans)

    out_stats = (
        lf.group_by(["window_start", "From"])
        .agg(
            pl.len().alias("deg_out"),
            pl.col("To").n_unique().alias("fan_out"),
            pl.col("Amount").sum().alias("vol_out"),
        )
        .rename({"From": "Node"})
    )

    in_stats = (
        lf.group_by(["window_start", "To"])
        .agg(
            pl.len().alias("deg_in"),
            pl.col("From").n_unique().alias("fan_in"),
            pl.col("Amount").sum().alias("vol_in"),
        )
        .rename({"To": "Node"})
    )

    ego = (
        out_stats.join(in_stats, on=["window_start", "Node"], how="full")
        .with_columns(
            pl.col("deg_out").fill_null(0).cast(pl.Int64),
            pl.col("fan_out").fill_null(0).cast(pl.Int64),
            pl.col("vol_out").fill_null(0.0),
            pl.col("deg_in").fill_null(0).cast(pl.Int64),
            pl.col("fan_in").fill_null(0).cast(pl.Int64),
            pl.col("vol_in").fill_null(0.0),
        )
        .with_columns(
            ((pl.col("vol_in") - pl.col("vol_out"))/ (pl.col("vol_in") + pl.col("vol_out") + pl.lit(EPS))).alias("flow_imbalance")
        )
        .with_columns(
            pl.when(pl.col('window_start').is_null()).then(pl.col('window_start_right')).otherwise(pl.col('window_start')).alias('window_start'),
            pl.when(pl.col('Node').is_null()).then(pl.col('Node_right')).otherwise(pl.col('Node')).alias('Node'),
        )
        .drop('Node_right', 'window_start_right')
    )

    return ego.collect()

In [23]:
ego_profile = compute_ego_profiles(ssl)

In [24]:
ego_profile

window_start,Node,deg_out,fan_out,vol_out,deg_in,fan_in,vol_in,flow_imbalance
datetime[μs],u32,i64,i64,f64,i64,i64,f64,f64
2022-09-15 00:00:00,176949,0,0,0.0,7,2,120504.655334,1.0
2022-09-11 00:00:00,19166,5,2,582778.498862,5,2,1255.770736,-0.9957
2022-09-07 00:00:00,848725,14,3,14589.70668,4,1,9273.947385,-0.222755
2022-09-11 00:00:00,982784,0,0,0.0,2,1,655.02,1.0
2022-09-07 00:00:00,399913,0,0,0.0,6,2,1822.592763,1.0
…,…,…,…,…,…,…,…,…
2022-09-09 00:00:00,1707698,1,1,68.39,0,0,0.0,-1.0
2022-09-01 00:00:00,654387,1,1,14309.946896,0,0,0.0,-1.0
2022-09-03 00:00:00,1543126,1,1,3629.1,0,0,0.0,-1.0


# Flow-aware positional prediction

In [25]:
EPS = 1e-12

def flow_targets_out_entropy_amount_fast(trans: DataFrame, k2: bool = True) -> DataFrame:
    lf = _prep(trans).select(["window_start", "From", "To", "Amount"]).lazy()

    W = (
        lf.group_by(["window_start", "From", "To"])
        .agg(pl.col("Amount").sum().alias("w"))
    )

    P = (
        W.with_columns(
            pl.col("w").sum().over(["window_start", "From"]).alias("out_wsum")
        )
        .with_columns(
            (pl.col("w") / (pl.col("out_wsum") + pl.lit(EPS))).alias("p")
        )
        .select(["window_start", "From", "To", "p"])
        .cache()
    )

    one = (
        P.group_by(["window_start", "From"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_1_amt"),
            pl.len().alias("supp_out_1_amt"),
            pl.col("p").max().alias("pmax_out_1_amt"),
        )
        .rename({"From": "Node"})
    )

    if not k2:
        return one.collect(streaming=True)

    P1 = P.rename({"From": "src", "To": "mid", "p": "p1"})
    P2 = P.rename({"From": "mid", "To": "dst", "p": "p2"})

    two_step = (
        P1.join(P2, on=["window_start", "mid"], how="inner")
        .with_columns((pl.col("p1") * pl.col("p2")).alias("p"))
        .group_by(["window_start", "src", "dst"])
        .agg(pl.col("p").sum().alias("p"))
    )

    two = (
        two_step.group_by(["window_start", "src"])
        .agg(
            (-pl.col("p") * (pl.col("p") + pl.lit(EPS)).log()).sum().alias("H_out_2_amt"),
            pl.len().alias("supp_out_2_amt"),
            pl.col("p").max().alias("pmax_out_2_amt"),
        )
        .rename({"src": "Node"})
    )

    return one.join(two, on=["window_start", "Node"], how="left").collect(streaming=True)


In [ ]:
train_pos_pred = flow_targets_out_entropy_amount_fast(ssl)

/var/folders/0x/klqdyhpj427dvg81kxzyfcvm0000gn/T/ipykernel_27968/1113959802.py:144: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  return one.join(two, on=["window_start", "Node"], how="left").collect(streaming=True)


# Join

In [ ]:
node_features = (
    temporal_features
    .join(train_pos_pred, on=["window_start", "Node"], how="full", coalesce=True)
    .join(ego_profile, on=["window_start", "Node"], how="full", coalesce=True)
    .join(two_cycles, on=["window_start", "Node"], how="full", coalesce=True)
    .with_columns(
        (pl.col("window_start") - pl.col("window_start").min())
        .dt.total_seconds()
        .cast(pl.Int64)
        .add(10)
        .alias("window_start"),

        pl.lit(1).alias("Feature"),
    )
    .sort("window_start")
)
node_features

In [28]:
trans = (
    _prep(ssl).with_columns(
        (pl.col("Timestamp") - pl.col("Timestamp").min())
        .dt.total_seconds()
        .cast(pl.Int64)
        .add(10)
        .alias("Timestamp"),
    )
    .with_columns(
        (pl.col("window_start") - pl.col("window_start").min())
        .dt.total_seconds()
        .cast(pl.Int64)
        .add(10)
        .alias("window_start"),

        pl.lit(1).alias("Feature"),
    )
    .sort("window_start")
    .with_row_index("Edge ID")
    .collect()
)
trans.head()

Edge ID,Timestamp,From Bank,To Bank,Is Laundering,Amount,From,To,Payment Format,window_start,Feature
u32,i64,i64,i64,i64,f64,u32,u32,u32,i64,i32
0,10,1046,274159,0,26.42,1997667,581807,0,10,1
1,10,21418,21418,0,23.61,1755234,1755234,6,10,1
2,10,32248,11,0,12373.74,1077771,2067545,5,10,1
3,10,11798,11798,0,4981.27,552140,552140,6,10,1
4,10,1924,1924,0,22.94,519381,519381,6,10,1


# Write

In [ ]:
node_features.write_csv('../data/HI-Medium_SSL_Nodes_2d_convert.csv')
trans.write_csv('../data/HI-Medium_SSL_Trans_2d_convert.csv')